In [7]:
import pandas as pd

url = "https://huggingface.co/datasets/chuso6/har_features/resolve/main/har_features.csv"

df_final_features = pd.read_csv(url)
df_final_features.head()

,activity,subject,window_id,T_acc_mean_x,T_acc_mean_y,T_acc_mean_z,T_acc_std_x,T_acc_std_y,T_acc_std_z,T_acc_max_x,...,LL_mag_max_x,LL_mag_max_y,LL_mag_max_z,LL_mag_corr_xy,LL_mag_corr_xz,LL_mag_corr_yz,LL_mag_mag_mean,LL_mag_mag_std,LL_mag_mag_auc,LL_mag_mag_mean_diff
0,1,1,0,8.015509,1.058076,5.553903,0.129444,0.039797,0.191729,8.1605,...,0.74182,0.30267,-0.055365,-0.380922,0.214412,-0.094971,0.800508,0.000745,2.369513,0.000941
1,1,1,1,7.920071,1.126935,5.683707,0.058170,0.026639,0.105984,8.0412,...,0.74320,0.30342,-0.054963,-0.351583,0.448888,-0.306916,0.801040,0.000701,2.371086,0.000761
2,1,1,2,8.001183,1.141395,5.559029,0.095242,0.030720,0.148445,8.1763,...,0.74335,0.30377,-0.054945,-0.231525,0.377849,-0.342104,0.801930,0.000862,2.373707,0.000829
3,1,1,3,7.941989,1.143843,5.658659,0.059360,0.024328,0.094272,8.1160,...,0.74302,0.30397,-0.054711,-0.266598,0.365792,-0.255355,0.802269,0.000731,2.374725,0.000797
4,1,1,4,7.996011,1.138048,5.567233,0.042821,0.021047,0.067826,8.0860,...,0.74316,0.30423,-0.055413,-0.169517,0.621999,-0.270246,0.802356,0.000820,2.375000,0.000938


In [8]:
print(df_final_features.columns)

Index(['activity', 'subject', 'window_id', 'T_acc_mean_x', 'T_acc_mean_y',
       'T_acc_mean_z', 'T_acc_std_x', 'T_acc_std_y', 'T_acc_std_z',
       'T_acc_max_x',
       ...
       'LL_mag_max_x', 'LL_mag_max_y', 'LL_mag_max_z', 'LL_mag_corr_xy',
       'LL_mag_corr_xz', 'LL_mag_corr_yz', 'LL_mag_mag_mean', 'LL_mag_mag_std',
       'LL_mag_mag_auc', 'LL_mag_mag_mean_diff'],
      dtype='object', length=243)


##Nuevo AdaBoost

In [9]:
import numpy as np
import pandas as pd
import time

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA, PCA
from sklearn.feature_selection import SelectFromModel

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# =========================================================
# GRIDS DE HIPERPARÁMETROS
# =========================================================

GRID_KPCA = [
    {
        'reduccion__kernel': ['rbf'],
        'reduccion__n_components': [32, 64, 128],
        'reduccion__gamma': [0.001, 0.01, 0.1, 'scale']
    },
    {
        'reduccion__kernel': ['poly'],
        'reduccion__n_components': [32, 64],
        'reduccion__degree': [2, 3],
        'reduccion__gamma': ['scale']
    }
]

GRID_ADABOOST = {
    'modelo__learning_rate': [0.001, 0.01, 0.1, 0.5, 1.0, 2.0],
    'modelo__n_estimators': [25, 50, 100]
}

# =========================================================
# NESTED LOSO — ADABOOST
# =========================================================

def nested_loso_adaboost(
    datos,
    nombre_reduccion,          # 'pca' | 'kpca' | 'bosque' | 'sin'
    target_col="activity",
    group_col="subject",
    scoring="accuracy",
    average="weighted",
    random_state=42
):
    print(f"--- NESTED LOSO | Modelo: ADABOOST | Reducción: {nombre_reduccion.upper()} ---")

    X = datos.drop(columns=[target_col, group_col])
    y = datos[target_col]
    groups = datos[group_col]

    logo_outer = LeaveOneGroupOut()
    fold = 1

    accuracies, precisions, recalls, f1_scores = [], [], [], []
    start_time_total = time.time()

    for train_index, test_index in logo_outer.split(X, y, groups):
        start_time_fold = time.time()

        X_test      = X.iloc[test_index]
        y_test      = y.iloc[test_index]
        sujeto_eval = groups.iloc[test_index].unique()[0]

        X_train_val      = X.iloc[train_index]
        y_train_val      = y.iloc[train_index]
        groups_train_val = groups.iloc[train_index]

        # =====================================================
        # 1. PIPELINE + GRID SEGÚN REDUCCIÓN
        # =====================================================

        pasos = [('scaler', StandardScaler())]

        if nombre_reduccion == 'pca':
            pasos.append(('reduccion', PCA(n_components=0.95, random_state=random_state)))
            param_grid_red = {}

        elif nombre_reduccion == 'kpca':
            pasos.append(('reduccion', KernelPCA(random_state=random_state, n_jobs=-1)))
            param_grid_red = GRID_KPCA

        elif nombre_reduccion == 'bosque':
            selector = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
            pasos.append(('reduccion', SelectFromModel(selector)))
            param_grid_red = {}

        elif nombre_reduccion == 'sin':
            param_grid_red = {}

        else:
            raise ValueError(f"Reducción '{nombre_reduccion}' no implementada.")

        pasos.append((
            'modelo',
            AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1),
                random_state=random_state
            )
        ))

        pipeline = Pipeline(pasos)

        # =====================================================
        # 2. COMBINAR GRIDS
        # =====================================================

        if isinstance(param_grid_red, list):
            param_grid = [{**bloque, **GRID_ADABOOST} for bloque in param_grid_red]
        else:
            param_grid = {**param_grid_red, **GRID_ADABOOST}

        # =====================================================
        # 3. INNER CV + FIT
        # =====================================================

        inner_cv = GroupKFold(n_splits=3)

        search = GridSearchCV(
            pipeline,
            param_grid,
            cv=inner_cv,
            scoring=scoring,
            n_jobs=-1
        )

        search.fit(X_train_val, y_train_val, groups=groups_train_val)

        mejor_pipeline = search.best_estimator_
        params_limpios = {k.split('__')[1]: v for k, v in search.best_params_.items()}

        # =====================================================
        # 4. PREDICCIÓN
        # =====================================================

        y_pred = mejor_pipeline.predict(X_test)

        # =====================================================
        # 5. MÉTRICAS + PRINT
        # =====================================================

        acc_fold = accuracy_score(y_test, y_pred)
        f1_fold  = f1_score(y_test, y_pred, average=average, zero_division=0)

        accuracies.append(acc_fold)
        precisions.append(precision_score(y_test, y_pred, average=average, zero_division=0))
        recalls.append(recall_score(y_test, y_pred, average=average, zero_division=0))
        f1_scores.append(f1_fold)

        print(
            f"[Iter {fold}] Test Sujeto {sujeto_eval} -> "
            f"Params: {params_limpios} | "
            f"Acc: {acc_fold*100:.2f}% | "
            f"F1: {f1_fold*100:.2f}% | "
            f"Tiempo: {time.time() - start_time_fold:.2f} s"
        )

        fold += 1

    # =========================================================
    # RESULTADOS FINALES
    # =========================================================

    print(f"\n==========================================")
    print(f"Tiempo total LOSO Anidado: {(time.time() - start_time_total) / 60:.2f} min")
    print(f"ACCURACY FINAL : {np.mean(accuracies) * 100:.2f}% (± {np.std(accuracies) * 100:.2f}%)")
    print(f"PRECISION FINAL: {np.mean(precisions) * 100:.2f}%")
    print(f"RECALL FINAL   : {np.mean(recalls) * 100:.2f}%")
    print(f"F1-SCORE FINAL : {np.mean(f1_scores) * 100:.2f}%")

    return {
        "accuracy":  np.mean(accuracies),
        "precision": np.mean(precisions),
        "recall":    np.mean(recalls),
        "f1_score":  np.mean(f1_scores),
    }

In [10]:
resultados_pca = nested_loso_adaboost(
    datos=df_final_features,
    nombre_reduccion='bosque'
)

--- NESTED LOSO | Modelo: ADABOOST | Reducción: BOSQUE ---
[Iter 1] Test Sujeto 1 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 66.05% | F1: 60.96% | Tiempo: 1338.67 s
[Iter 2] Test Sujeto 2 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 53.53% | F1: 48.03% | Tiempo: 1322.34 s
[Iter 3] Test Sujeto 3 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 49.74% | F1: 46.60% | Tiempo: 1282.32 s
[Iter 4] Test Sujeto 4 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 60.63% | F1: 54.88% | Tiempo: 1320.57 s
[Iter 5] Test Sujeto 5 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 35.11% | F1: 29.09% | Tiempo: 1312.30 s
[Iter 6] Test Sujeto 6 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 39.00% | F1: 31.07% | Tiempo: 1324.94 s
[Iter 7] Test Sujeto 7 -> Params: {'learning_rate': 2.0, 'n_estimators': 100} | Acc: 47.47% | F1: 40.62% | Tiempo: 1327.44 s
[Iter 8] Test Sujeto 8 -> Params: {'learning_rate': 2.0, 'n_estima